# Семинар 5. Когда AI-подсказка помогает, а когда мешает?

Проект: определить интернет-цену товара по странице магазина.

## План семинара

1. Правило выбора цены и устройство пайплайна.
2. Готовый эксперимент: что меняется при показе подсказки?
3. Аналитическое отступление.
4. Пытаемся улучшить результат.
5. Неопределенность и выводы.

## Как устроен семинар

**Часть I. Разбираем готовый пайплайн.** URL → скриншот → VLM → подсказка → Яндекс Задания.
Смотрим настройки, промпт и интерфейс проведенного запуска. Код доступен для самостоятельного повторения.
Вопрос этой части: **как построить подсказку и показать её исполнителю?**

**Часть II. Исследуем готовую выгрузку.** 100 карточек, две группы, 600 ответов.
Для исследования использован заранее подготовленный набор подсказок с контролируемой долей ошибок.
Вопрос этой части: **что подсказка меняет в человеческих ответах?**

**Часть III. Меняем промпт онлайн: стало ли лучше?** На тех же сохраненных скриншотах реально сравниваем V1 и V2.
Вопрос этой части: **поможет ли более подробная инструкция уменьшить риск ошибок?**

Далее **Baseline** — группа без подсказки, **AI hint** — группа с подсказкой;
**gold** — проверенная цена по нашей инструкции. V1 и V2 — версии промпта, а не группы исполнителей.

### Три гипотезы

- **H1 — помощь:** на карточках с верной подсказкой правильность ответа человека выше.
- **H2 — влияние ошибочной подсказки:** на карточках с ошибочной подсказкой люди чаще выбирают именно предложенную неверную цену.
- **H3 — общий результат:** баланс помощи и вреда зависит от состава подсказок.

## Окружение

Структура: `requirements.txt`, ноутбук, `src/` с вспомогательными модулями и `data/` с данными.
`src/workbook_capture.py` снимает сайты; `workbook_model.py` запускает VLM;
`workbook_support.py` работает с API; `seminar5_research.py` считает метрики;
`prompt_study.py` сохраняет и сравнивает реальные модельные прогоны.

В Colab загрузите папку семинара или клонируйте репозиторий курса и перейдите в неё.
Установка: `!pip install -r requirements.txt`. Для новых скриншотов: `!playwright install --with-deps chromium`.
Для VLM удобнее GPU; CPU работает медленнее и требует достаточно RAM. Здесь нет автоматических платных запусков.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import sys, json, uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image as PILImage
from IPython.display import display, HTML, Markdown, Image

ROOT = Path.cwd() if (Path.cwd() / "data/seminar5_challenge").exists() else Path.cwd() / "shad_seminar"
sys.path.insert(0, str(ROOT / "src"))
BUNDLE = ROOT / "data/seminar5_challenge"
from prompt_study import TASK_RULE
from seminar5_research import load_experiment, cluster_interval
ARMS = ["baseline", "assisted"]
LABELS = ["Baseline", "AI hint"]
COLORS = ["#6479AF", "#D98559"]
plt.rcParams.update({"figure.dpi":120, "font.size":11, "axes.spines.top":False, "axes.spines.right":False})
cards = pd.read_csv(BUNDLE / "cards.csv", dtype={"sample_id":str})
display(Markdown(f"В выборке **{len(cards)} карточек**. Gold хранится в `reference_price`, цена подсказки — в `hint_price`."))

# Часть I. Как устроен пайплайн

## Данные и правило выбора цены

На одной странице встречаются цена обычного покупателя, опт, старая цена, платеж рассрочки
и другой вариант товара. Какая из них отвечает нашей инструкции?

In [ ]:
display(Markdown("**Краткое правило для модели и обсуждения (TASK_RULE):**"))
display(Markdown(TASK_RULE))
worker_instruction = (BUNDLE / "instruction.html").read_text()
display(HTML('<details><summary>Полная инструкция, которую видел исполнитель</summary>' + worker_instruction + '</details>'))
example = cards.set_index("sample_id").loc["7254e1e3decff0dd"]
display(Image(filename=str(ROOT / example.image_path), width=800))
display(Markdown("**Пример:** интернет-цена 47 990 ₽, цена в розничном магазине 54 300 ₽. Нужна интернет-цена."))

## Baseline и экспериментальный дизайн

Механику API разбирали на предыдущем семинаре. Здесь важен дизайн сравнения:

| Параметр | Baseline | AI hint |
|---|---|---|
| URL | Те же 100 | Те же 100 |
| Инструкция и оплата | Одинаковые | Одинаковые |
| Ответ | Цена / причина отсутствия ответа | Цена / причина отсутствия ответа |
| Подсказка | Нет | **Есть** |
| Исполнители | Отдельная группа | Отдельная группа |
| Перекрытие | 3 | 3 |

**Меняем одну вещь: наличие AI-подсказки.** Ответ человека не предзаполняем.
Ни gold, ни правильность подсказки исполнителю не передаем.

Ниже готовим конфигурацию для собственного запуска. Сам вызов платного API вынесен в `src/home_experiment.py`;
на занятии используем уже размеченную выгрузку. Для нового полноценного эксперимента нужно отдельно
сформировать непересекающиеся группы и проверить бюджет — два произвольных пула этого не гарантируют.

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("seminar5_%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
pool_settings = {
    "private_name": RUN_ID,
    "may_contain_adult_content": False,
    "will_expire": (datetime.now(timezone.utc) + timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%S"),
    "reward_per_assignment": 1.0,
    "assignment_max_duration_seconds": 600,
    "auto_accept_solutions": True,
    "mixer_config": {"real_tasks_count":1,"golden_tasks_count":0,"training_tasks_count":0},
    "defaults": {"default_overlap_for_new_task_suites":3},
}
baseline_project = json.loads((BUNDLE / "baseline_project.json").read_text())
display(pd.Series({"Перекрытие":3,"Оплата за ответ, ₽":1,"Срок до":pool_settings["will_expire"]}).to_frame("Настройка"))
print("Конфигурация подготовлена локально. Проект и пул не создавались.")

## От URL к AI-подсказке

**Обсудим до написания кода.** Человек получает URL: открывает сайт, находит товар, читает условия цен.
Подсказка могла бы сократить время поиска и часть переходов. Модель не испытывает человеческой усталости:
одинаковую инструкцию можно применять к каждому заданию без снижения внимания.
Но это **не гарантирует одинаковой точности**: качество зависит от страницы, мелкого текста и условий цены;
систематическую ошибку модель может повторять много раз.

Предложите устройство помощника: что передать модели, если на входе в задании только URL?
Как получить содержимое страницы? Что написать в подсказке? Что должен проверить человек?

HTML может содержать много меню и служебного кода, а извлечение текста — потерять связь цены с подписью.
Скриншот сохраняет расположение, но мелкий шрифт может потеряться при уменьшении.
Изображение не всегда дешевле или точнее текста.


Откройте пример по ссылке, затем выполните ячейку захвата. Браузер работает без входа в аккаунт.
Если сайт вернул ограничение доступа, не обходим его: выбираем другую публичную страницу.

In [ ]:
from playwright.async_api import async_playwright
from urllib.parse import urlsplit
import html

async def screenshot_site(url, output_dir):
    """Открываем страницу в отдельном браузере и сохраняем видимую область."""
    parsed = urlsplit(url)
    if parsed.scheme != "https" or not parsed.hostname or parsed.username:
        raise ValueError("Нужен публичный HTTPS-адрес магазина")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"page_{uuid.uuid4().hex[:8]}.png"
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        try:
            page = await browser.new_page(viewport={"width":1440,"height":1400}, locale="ru-RU")
            response = await page.goto(url, wait_until="domcontentloaded", timeout=45000)
            if response is None or response.status >= 400:
                raise RuntimeError(f"Страница недоступна: HTTP {response.status if response else 'нет ответа'}")
            await page.locator("body").wait_for(state="visible")
            await page.wait_for_timeout(2000)  # Даём динамическим блокам появиться.
            await page.screenshot(path=str(path), full_page=False)
        finally:
            await browser.close()
    return path

example_url = example.source_url
display(HTML(f'<a href="{html.escape(example_url, quote=True)}" target="_blank" rel="noopener noreferrer">Открыть страницу товара ↗</a>'))

In [ ]:
screenshot_path = await screenshot_site(example_url, ROOT / "data/classroom_captures")
display(Image(filename=str(screenshot_path), width=700))

In [ ]:
def prepare_vlm_input(image_path, prompt):
    """RGB-изображение и сообщение для локальной VLM; URL ей уже не нужен."""
    with PILImage.open(image_path) as source:
        picture = source.convert("RGB")
    messages = [{"role":"user", "content":[
        {"type":"image"},
        {"type":"text", "text":prompt},
    ]}]
    return picture, messages

### Практическое задание №1. Напишите минимальный промпт для VLM

Модель получает скриншот карточки товара. Нужно извлечь цену для обычного покупателя
и вернуть только JSON с полями `price` и `status`:

```json
{"price": 123.45, "status": "ok"}
```

Допустимые статусы: `ok`, `no_price`, `uncertain`. Для `ok` цена положительна,
для остальных — `null`. 

Начните с короткой версии. Какую часть человеческой инструкции `TASK_RULE` необходимо передать уже сейчас?

In [ ]:
# 1. Опишите товар, нужную цену и формат price/status.
PROMPT_V1 = ...  # YOUR CODE HERE: строка с коротким промптом
print(PROMPT_V1)

In [ ]:
def render_hint(parsed):
    if parsed["status"] == "ok":
        amount = f'{parsed["price"]:,.2f}'.replace(",", " ").replace(".", ",")
        return f"Цена товара — {amount} ₽."
    if parsed["status"] == "no_price":
        return "На странице цена не указана."
    return "Не удалось определить цену."

## Добавляем подсказку

Вход baseline: `sample_id`, `source_url`. Во второй группе добавляется только `hint`.
Ссылка остаётся доступной, числовой ответ не предзаполнен. Ниже — настоящий скрин предпросмотра интерфейса;
число в поле ответа введено при проверке, а не подставлено автоматически.

In [ ]:
display(Image(filename=str(ROOT / "assets/ai_hint_ui_card.jpg"), width=450))

In [ ]:
assisted_project = json.loads((BUNDLE / "assisted_project.json").read_text())
base_fields = set(baseline_project["task_spec"]["input_spec"])
ai_fields = set(assisted_project["task_spec"]["input_spec"])
print("Baseline:", ", ".join(sorted(base_fields)))
print("AI hint добавляет:", ", ".join(sorted(ai_fields - base_fields)))

**Как отправить подсказку вместе с заданием.** В проекте объявлено входное поле `hint`,
а интерфейс показывает его в блоке AI-подсказки. Для каждого URL текст передаем в `input_values`.
Ниже весь пример создания проекта, пула и заданий; функция сама по себе ничего не запускает.

Для домашнего запуска нужны свой токен, баланс и фильтр аудитории. Перед подтверждением проверьте
в кабинете интерфейс и бюджет с комиссией. Для сравнения двух групп дополнительно нужны
непересекающиеся аудитории; один этот пример их не формирует. Модерация может запустить оплату автоматически.

In [ ]:
def launch_assisted_pool(tasks_with_hints, audience_filter, budget_rub,
                         run_id, reward_rub=1.0, overlap=3, commission_rate=0.30):
    """Платный запуск. Вход: DataFrame с sample_id, source_url, hint."""
    from getpass import getpass
    from workbook_support import TasksAPI
    from home_experiment import request_launch
    import copy

    required = ["sample_id","source_url","hint"]
    rows = tasks_with_hints[required].to_dict("records")
    if not rows or tasks_with_hints.sample_id.duplicated().any():
        raise ValueError("Нужны уникальные задания")
    if not isinstance(audience_filter,dict) or not audience_filter:
        raise ValueError("Укажите проверенный фильтр аудитории")
    if any(not isinstance(row[k],str) or not row[k].strip() for row in rows for k in required):
        raise ValueError("Заполните ID, URL и текст подсказки")
    if reward_rub <= 0 or type(overlap) is not int or overlap < 1 or commission_rate < 0:
        raise ValueError("Проверьте оплату, перекрытие и комиссию")
    estimate = len(rows) * overlap * reward_rub * (1 + commission_rate)
    if not np.isfinite(budget_rub) or estimate > budget_rub:
        raise ValueError(f"Смета {estimate:.2f} ₽ превышает разрешенный бюджет")

    # Проект содержит инструкцию, схему ответа и готовый UI с AI-подсказкой.
    project_spec = copy.deepcopy(assisted_project)
    project_spec["public_name"] = "Найдите интернет-цену товара"
    project_spec["task_spec"]["input_spec"]["hint"] = {"type":"string","required":True,"hidden":False}

    # Настройки пула: оплата за страницу из одного задания, срок, перекрытие и аудитория.
    pool_spec = {
        "private_name":run_id + " / AI hint",
        "may_contain_adult_content":False,
        "will_expire":(datetime.now(timezone.utc)+timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%S"),
        "reward_per_assignment":reward_rub,
        "assignment_max_duration_seconds":600,
        "auto_accept_solutions":True,
        "mixer_config":{"real_tasks_count":1,"golden_tasks_count":0,"training_tasks_count":0},
        "defaults":{"default_overlap_for_new_task_suites":overlap},
        "filter":audience_filter,
    }
    api = TasksAPI(getpass("OAuth-токен Яндекс Заданий: ").strip(),
                   ROOT / "data/home_runs" / run_id / "api_journal.json",allow_write=True)
    project = api.write_once("project","POST","/api/v1/projects",project_spec)
    pool_spec["project_id"] = project["id"]
    pool = api.write_once("pool","POST","/api/v1/pools",pool_spec)

    # Именно здесь текст подсказки привязывается к конкретному URL.
    tasks = [{"pool_id":pool["id"],"overlap":overlap,
              "input_values":{"sample_id":row["sample_id"],"source_url":row["source_url"],"hint":row["hint"]}}
             for row in rows]
    api.write_once("tasks","POST","/api/v1/tasks",tasks,
                   params={"async_mode":"false","skip_invalid_items":"false","open_pool":"false"})
    print("Создан закрытый пул:",pool["id"],"; расчетная смета:",round(estimate,2),"₽")
    # Валидация, ручное START и отправка на модерацию либо открытие готового пула.
    request_launch(api,pool["id"],budget_rub)
    return pool["id"]

# Пример входа (после generate_prices): соедините sample_id/source_url с рассчитанным hint.
# tasks_with_hints = pd.DataFrame([{ "sample_id":..., "source_url":..., "hint":"Цена товара — ... ₽." }])
# audience_filter = ...  # Фильтр вашей аудитории из настроек пула.
# pool_id = launch_assisted_pool(tasks_with_hints,audience_filter,budget_rub=...,run_id=RUN_ID)
# Комиссию сверяем в кабинете; 30% здесь — расчетный параметр, не гарантированный тариф.

# Часть II. Что изменилось в ответах людей?

Сравним две группы на одних и тех же карточках: три ответа на карточку в каждой группе.
Качество подсказки определяем сравнением с gold — проверенной ценой по нашей инструкции.

In [ ]:
data, cards, metadata = load_experiment(BUNDLE)
counts = data.groupby(["sample_id","condition"]).size()
assert len(counts) == 2 * len(cards) and counts.eq(3).all()
assert data.groupby("worker").condition.nunique().eq(1).all()
assert not data.duplicated(["worker","sample_id"]).any()
design = data.groupby("condition").agg(Карточек=("sample_id","nunique"), Ответов=("sample_id","size"), Исполнителей=("worker","nunique"))
display(design.reindex(ARMS).rename(index=dict(zip(ARMS,LABELS))))

# Считаем по карточкам, не по повторным ответам исполнителей.
hint_is_correct = (cards["hint_price"] - cards["reference_price"]).abs() < .005
hint_counts = pd.Series({"Совпала с gold":int(hint_is_correct.sum()), "Не совпала с gold":int((~hint_is_correct).sum())})
hint_overview = hint_counts.to_frame("Карточек")
hint_overview["Доля выборки"] = hint_counts / len(cards)
display(Markdown("**Качество подсказок до сравнения ответов людей**"))
display(hint_overview.style.format({"Доля выборки":"{:.0%}"}))

## Главный вопрос: чаще ли люди выбирают ошибочную цену из подсказки?

Считаем долю людей, назвавших **конкретную неверную цену из подсказки**, а не любую неверную цену.
Для каждой группы отдельно:

$$P(\text{ответ человека}=\text{подсказка}\mid\text{подсказка неверна}).$$

Короткая запись: $P(H=A\mid A\ne Y)$, где $H$ — ответ человека, $A$ — подсказка, $Y$ — gold.
Главный эффект — разница этой доли между AI hint и Baseline, в процентных пунктах.

<details><summary>Как посчитать оценку по ответам</summary>

$$U_g=\frac{\sum_{i,j:G_j=g,A_i\ne Y_i}\mathbb1[H_{ij}=A_i]}{N_{g,\mathrm{wrong}}},
\qquad\Delta U=U_{\mathrm{AI}}-U_{\mathrm{baseline}}.$$

</details>

В baseline человек **не видел подсказку**. Но мог сам выбрать ту же заметную неверную цену.
Сравниваем именно эту ошибку на одинаковых карточках, а не любые ошибки.
Знаменатель — все 99 ответов каждой группы на 33 карточках, включая отказы.

Совпадение не доказывает копирования конкретным человеком: это разные люди в разных группах,
а не ответ одного человека «до и после». Межгрупповая разница оценивает эффект показа подсказки.

**Практика №2. Совпадение с ошибочной подсказкой**

1. `wrong`: оставьте строки, где `hint_wrong == True`.
2. `wrong_hint_match_rate`: среднее булевого `matches_hint` отдельно по `condition`.
3. `excess_wrong_hint_agreement`: доля `assisted` минус доля `baseline`.

Не удаляйте отказы: они остаются в знаменателе. Следующая ячейка нарисует именно эти две доли.
Почему совпадение может быть ненулевым в Baseline, где подсказку не показывали?

In [ ]:
# 1. Отбираем ответы на карточках с неверной подсказкой.
wrong = ...  # YOUR CODE HERE: фильтр по hint_wrong
# 2. True = 1, False = 0: среднее даёт долю совпадений.
wrong_hint_match_rate = ...  # YOUR CODE HERE: среднее matches_hint по condition
# 3. Вычитаем Baseline из AI hint (пока в долях, не в процентах).
excess_wrong_hint_agreement = ...  # YOUR CODE HERE: assisted − baseline

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
rates_percent = 100 * wrong_hint_match_rate.reindex(ARMS)
bars = ax.bar(LABELS, rates_percent, color=COLORS)
ax.bar_label(bars, labels=[f"{value:.1f}%" for value in rates_percent], padding=5)
ax.set(ylabel="Совпадение с неверной подсказкой, %", ylim=(0,max(35,rates_percent.max()*1.3)),
       title=f"Прирост совпадения: {100*excess_wrong_hint_agreement:+.1f} п.п.")
plt.show()
display(Markdown("Сравниваем выбор **одной и той же неверной цены**. Межгрупповая разница не доказывает копирования конкретным человеком."))

**Достаточно ли убедителен прирост?** Один исполнитель мог ответить на несколько карточек.
Поэтому строим 95% кластерный bootstrap-интервал по исполнителям внутри каждой группы,
оставляя выборку карточек фиксированной. Если весь интервал выше нуля, рост статистически
убедителен по этому приближенному критерию. Это не устанавливает психологический механизм ошибки.

In [ ]:
wrong_match_interval = cluster_interval(data,"matches_hint",subset="wrong")
np.testing.assert_allclose(wrong_match_interval["effect"],excess_wrong_hint_agreement)
effect_pp = 100 * excess_wrong_hint_agreement
low_pp = 100 * wrong_match_interval["low"]
high_pp = 100 * wrong_match_interval["high"]
significant_increase = low_pp > 0

fig, ax = plt.subplots(figsize=(9,2.6))
ax.hlines(0,low_pp,high_pp,color=COLORS[0],lw=5)
ax.scatter(effect_pp,0,color=COLORS[1],s=90,zorder=3)
ax.axvline(0,color="black",ls="--",label="Нет изменения")
ax.set(yticks=[],xlabel="Прирост совпадения с неверной подсказкой, п.п.",
       title=f"Оценка {effect_pp:+.1f}; 95% интервал [{low_pp:+.1f}; {high_pp:+.1f}]")
ax.legend(); plt.show()
inference = ("Прирост статистически убедителен: весь интервал выше нуля."
             if significant_increase else "Убедительный положительный эффект по этому интервалу не установлен.")

## Помощь и вред

Теперь проверим качество. Сначала вред на карточках с неверной подсказкой, затем помощь верных
и итог на всем пуле. В baseline названия подвыборок определяются подсказкой, показанной другой группе.

Точная цена — совпадение с gold с допуском меньше половины копейки. Отказы остаются в знаменателе.
Внутри каждого среза сравниваем две группы людей, а не разные карточки между собой.

**Практика №3. Посчитаем помощь и вред отдельно**

Здесь `correct` означает правильность **ответа человека**, а `hint_wrong` — ошибочность **подсказки**.

1. Создайте `correct_hint_answers` и `wrong_hint_answers` — два среза по `hint_wrong`.
2. Для каждого среза получите среднюю `correct` по `condition`: `accuracy_good` и `accuracy_bad`.
3. Соберите `quality_table`: строки «Верная подсказка» / «Неверная подсказка», столбцы `baseline` / `assisted`.
4. `quality_delta_pp`: разность столбцов, умноженная на 100.

Код ниже построит таблицу и график из этих переменных.
Насколько помогают верные подсказки? Насколько вредят неверные? От чего зависит общий результат?

In [ ]:
# 1. Разделяем ответы по качеству подсказки, не по ответу человека.
correct_hint_answers = ...  # YOUR CODE HERE: hint_wrong == False
wrong_hint_answers = ...  # YOUR CODE HERE: hint_wrong == True
# 2. В каждом срезе считаем долю правильных ответов по двум группам.
accuracy_good = ...  # YOUR CODE HERE: средняя correct по condition
accuracy_bad = ...  # YOUR CODE HERE: то же для неверных подсказок
# 3. Каждая Series становится строкой таблицы; столбцы уже называются condition.
quality_table = pd.DataFrame([accuracy_good,accuracy_bad], index=["Верная подсказка","Неверная подсказка"])[ARMS]
# 4. Переводим разницу долей в процентные пункты.
quality_delta_pp = ...  # YOUR CODE HERE: 100 × (assisted − baseline)

In [ ]:
display(quality_table.rename(columns=dict(zip(ARMS,LABELS))).style.format("{:.1%}"))
fig, axes = plt.subplots(1,2,figsize=(12,4),layout="constrained")
(quality_table.rename(columns=dict(zip(ARMS,LABELS))) * 100).plot.bar(ax=axes[0],color=COLORS,rot=0)
axes[0].set(ylabel="Правильных ответов, %", ylim=(0,110), title="Помощь и вред на разных карточках")
for bars in axes[0].containers:
    axes[0].bar_label(bars,fmt="%.1f",padding=3)
quality_delta_pp.plot.barh(ax=axes[1],color=["#3A9C89","#D76D67"])
axes[1].axvline(0,color="black",lw=1)
axes[1].set(xlabel="AI hint − Baseline, п.п.",title="Изменение правильности")
plt.show()

Общий результат — взвешенная сумма эффектов на двух подвыборках:

$$\Delta Q = w_{\mathrm{correct}}\Delta Q_{\mathrm{correct}} + w_{\mathrm{wrong}}\Delta Q_{\mathrm{wrong}}.$$

Веса — доли карточек с верной и неверной подсказкой. В коде это `weights`,
вклады — `contributions_pp`, их сумма — `overall_delta_pp`.
Поэтому результат на этой выборке нельзя переносить на другие сайты и другую частоту ошибок без проверки.

In [ ]:
# 1. Веса — доли карточек с верной и неверной подсказкой.
weights = pd.Series([hint_is_correct.mean(),1-hint_is_correct.mean()],index=quality_table.index)
# 2. Эффекты на двух подвыборках уже найдены в практике №3.
contributions_pp = weights * quality_delta_pp
# 3. Складываем вклады в общую разницу.
overall_delta_pp = contributions_pp.sum()
decomposition = pd.DataFrame({"Вес":weights,"Эффект, п.п.":quality_delta_pp,"Вклад, п.п.":contributions_pp})
display(decomposition.style.format({"Вес":"{:.0%}","Эффект, п.п.":"{:+.1f}","Вклад, п.п.":"{:+.1f}"}))
terms = [f"{weight:.2f} × ({effect:+.2f})" for weight,effect in zip(weights,quality_delta_pp)]
display(Markdown("**Общий эффект:** " + " + ".join(terms) + f" = **{overall_delta_pp:+.2f} п.п.**"))

# 4. Независимая проверка через прямой подсчет accuracy на всей выборке.
overall_accuracy = data.groupby("condition")["correct"].mean().reindex(ARMS)
direct_delta_pp = 100 * (overall_accuracy["assisted"]-overall_accuracy["baseline"])
np.testing.assert_allclose(overall_delta_pp,direct_delta_pp)
fig, ax = plt.subplots(figsize=(8,3))
contributions_pp.plot.barh(ax=ax,color=["#3A9C89","#D76D67"])
ax.axvline(0,color="black",lw=1)
ax.set(xlabel="Вклад в изменение общей accuracy, п.п.",title=f"Вся выборка: {overall_delta_pp:+.1f} п.п.")
plt.show()
display(overall_accuracy.rename(index=dict(zip(ARMS,LABELS))).to_frame("Accuracy на всей выборке").style.format("{:.1%}"))

### Почему неверная подсказка убедительна?

Рассмотрим три примера: платеж по частям, оптовая цена и другой вариант товара.
В каждом типе выбираем первую карточку с неверной подсказкой по ID — **без отбора по ответам людей**.
Это иллюстрации разных источников правдоподобных ошибок, не репрезентативная подвыборка.
По трем ответам на одну карточку не делаем самостоятельного вывода об эффекте.

In [ ]:
case_types = {"installment":"Платёж по частям", "wholesale":"Оптовая цена", "variant":"Другой вариант товара"}
case_cards = cards.loc[~hint_is_correct].sort_values("sample_id")
for kind, title in case_types.items():
    card = case_cards.loc[case_cards.trap_type.eq(kind)].iloc[0]
    answers = data.loc[data.sample_id.eq(card.sample_id)]
    fig, axes = plt.subplots(1,2,figsize=(12,4),layout="constrained")
    with PILImage.open(ROOT / card.image_path) as picture:
        axes[0].imshow(picture)
    axes[0].axis("off")
    axes[0].set_title(title)
    for y, arm in enumerate(ARMS):
        group = answers.loc[answers.condition.eq(arm)]
        prices = group.loc[group.numeric,"price"].to_numpy()
        offsets = np.linspace(-.12,.12,len(prices))
        axes[1].scatter(prices/card.reference_price, y+offsets, s=70, color=COLORS[y])
        axes[1].text(.02,.95-y*.08,f"{LABELS[y]}: без цены {int((~group.numeric).sum())}",transform=axes[1].transAxes,va="top")
    axes[1].axvline(1,color="#3A9C89",label="Gold")
    axes[1].axvline(card.hint_price/card.reference_price,color="#D76D67",ls="--",label="Цена подсказки")
    axes[1].set(yticks=[0,1],yticklabels=LABELS,ylim=(-.4,1.6),xlabel="Ответ человека / gold",title="Каждая точка — ответ исполнителя")
    axes[1].legend(loc="lower right")
    plt.show()
    display(Markdown(f"Gold: **{card.reference_price:g} ₽**, подсказка: **{card.hint_price:g} ₽**. {card.reference_reason}"))

## Может ли перекрытие исправить ошибку?

Берём цену, которую назвали хотя бы двое из трех исполнителей. Без большинства не угадываем.
Совпавшие отказы не считаем большинством по цене.

Большинство полезно, когда ошибки людей достаточно независимы. Общая неверная подсказка может
направить несколько человек к одной ошибке.

> **Три ответа тогда не равны трем независимым проверкам:** общий источник подсказки может сделать ошибки зависимыми.

Поэтому проверяем не только наличие согласия, но и правильность итоговой цены.
Это объяснение возможного механизма, а не отдельное измерение корреляции ошибок.

**Практика №4. Агрегируем три ответа**

В функции `majority_price(group)` дана одна карточка одной группы:

1. `numeric_prices`: оставьте цены только строк с `numeric == True`.
2. `prices_in_kopecks`: умножьте на 100 и округлите до целого.
3. `vote_counts`: посчитайте, сколько раз встретилась каждая цена.
4. Верните лидирующую цену в рублях, если у неё минимум два голоса; иначе `np.nan`.

Готовый цикл применит функцию и создаст `majority_answers`. После этого:

5. `majority_answers['correct']`: сравните ответ с gold (разница меньше 0,005 ₽).
6. `majority_accuracy`: среднее `correct` по `condition`.

Нет большинства — неуспех, а не удалённая карточка. Gold не участвует в выборе цены.
График ниже использует `majority_answers` и `majority_accuracy`.

In [ ]:
def majority_price(group):
    # 1. Убираем отказы только из голосования, не из последующей оценки.
    numeric_prices = ...  # YOUR CODE HERE: цены числовых ответов
    # 2. Сравниваем целые копейки, а не float.
    prices_in_kopecks = ...  # YOUR CODE HERE: цены в целых копейках
    # 3. Самая частая цена будет первой.
    vote_counts = ...  # YOUR CODE HERE: частоты цен
    if vote_counts.empty or vote_counts.iloc[0] < 2:
        return np.nan
    # 4. Возвращаем рубли.
    return ...  # YOUR CODE HERE: цена-победитель в рублях

majority_rows = []
for (condition,sample_id),group in data.groupby(["condition","sample_id"]):
    majority_rows.append({"condition":condition,"sample_id":sample_id,"majority_price":majority_price(group)})
majority_answers = pd.DataFrame(majority_rows).merge(cards[["sample_id","reference_price"]],on="sample_id",validate="many_to_one")
# 5. NaN не совпадает с gold и даёт False.
majority_answers["correct"] = ...  # YOUR CODE HERE: abs(ответ − gold) < .005
# 6. Считаем по всем карточкам, включая отсутствие большинства.
majority_accuracy = ...  # YOUR CODE HERE: средняя correct по condition

In [ ]:
majority_answers["outcome"] = np.select(
    [majority_answers["correct"],majority_answers["majority_price"].isna()],
    ["Верное большинство","Нет большинства"],default="Ошибочное большинство")
majority_shares = pd.crosstab(majority_answers.condition,majority_answers.outcome,normalize="index")
majority_shares = majority_shares.reindex(index=ARMS,columns=["Верное большинство","Ошибочное большинство","Нет большинства"],fill_value=0)
np.testing.assert_allclose(majority_shares["Верное большинство"],majority_accuracy)
ax = majority_shares.rename(index=dict(zip(ARMS,LABELS))).plot.barh(stacked=True,color=["#3A9C89","#D76D67","#CBD1DD"],figsize=(9,3))
ax.set(xlim=(0,1),xlabel="Доля карточек",ylabel="",title="Что получилось после голосования?")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc="upper center",bbox_to_anchor=(.5,-.25),ncols=3,frameon=False)
plt.show()
display(majority_accuracy.rename(index=dict(zip(ARMS,LABELS))).to_frame("Accuracy большинства").style.format("{:.1%}"))

## Время и неопределенность

$$\Delta T = \frac{T_{\mathrm{AI}}-T_{\mathrm{Baseline}}}{T_{\mathrm{Baseline}}}.$$

$T$ — медиана времени от выдачи до отправки ответа: чтение, загрузка сайта, ввод и паузы.
Плюс означает дольше, минус — быстрее.

Но 300 ответов — не 300 независимых людей. Для интервала заново выбираем **исполнителей**
с возвращением внутри каждой группы и берём все их ответы. **Карточки считаем фиксированными.**
В каждом повторе снова считаем медианы и их относительное изменение; границы интервала — квантили 2,5% и 97,5%.
Если интервал пересекает ноль, данные совместимы и с ускорением, и с замедлением.

In [ ]:
# 1. Наблюдаемая разница медиан.
median_time = data.groupby("condition")["duration_sec"].median().reindex(ARMS)
time_change = (median_time["assisted"] - median_time["baseline"]) / median_time["baseline"]

# 2. Один элемент списка — все измеренные времена одного исполнителя.
worker_times = {}
for arm in ARMS:
    group = data.loc[data.condition.eq(arm)].dropna(subset=["duration_sec"])
    worker_times[arm] = [rows.duration_sec.to_numpy() for _,rows in group.groupby("worker")]

# 3. Повторно набираем исполнителей, а не отдельные ответы.
rng = np.random.default_rng(42)
bootstrap_time_changes = []
for _ in range(2000):
    sampled_medians = {}
    for arm in ARMS:
        clusters = worker_times[arm]
        selected = rng.integers(0,len(clusters),size=len(clusters))
        sampled_times = np.concatenate([clusters[i] for i in selected])
        sampled_medians[arm] = np.median(sampled_times)
    change = sampled_medians["assisted"] / sampled_medians["baseline"] - 1
    bootstrap_time_changes.append(change)

# 4. Центральные 95% повторных оценок.
time_low, time_high = np.quantile(bootstrap_time_changes,[.025,.975])
time_interval = {"effect":time_change,"low":time_low,"high":time_high}
includes_zero = time_low <= 0 <= time_high

fig, axes = plt.subplots(1,2,figsize=(12,3.5),layout="constrained")
bars = axes[0].bar(LABELS,median_time,color=COLORS)
axes[0].bar_label(bars,fmt="%.2f с",padding=4)
axes[0].set(ylabel="Секунды",title="Медиана в этой выборке",ylim=(0,median_time.max()*1.25))
axes[1].hlines(0,100*time_low,100*time_high,color="#6479AF",lw=5)
axes[1].scatter([100*time_change],[0],s=90,color="#D98559",zorder=3)
axes[1].axvline(0,color="black",ls="--",label="Нет изменения")
axes[1].set(yticks=[],xlabel="Изменение медианы, %",title="95% кластерный bootstrap-интервал")
axes[1].legend()
plt.show()
conclusion = ("Интервал включает 0: убедительного изменения времени не установлено."
              if includes_zero else "Интервал целиком выше 0: есть свидетельство замедления."
              if time_low > 0 else "Интервал целиком ниже 0: есть свидетельство ускорения.")
display(Markdown(f"**{conclusion}** Оценка {time_change:+.1%}, интервал [{time_low:+.1%}; {time_high:+.1%}]."))
if includes_zero:
    display(Markdown("**Итог:** подсказка изменила выбор ответов, но убедительной экономии времени мы не обнаружили. "
                     "Статистически убедительного увеличения времени тоже нет. Это не доказательство равенства времен "
                     "и не прямое наблюдение того, читал ли человек подсказку."))

# Часть III. Меняем промпт и проверяем качество

Запустим V1 на скриншотах golden set, разберем ошибки и попробуем V2.
Gold нужен только для оценки: в модель передаем изображение и промпт.
Для оценки используем сохранённые снимки: у свежей страницы цена могла измениться.

Можно несколько раз редактировать V2 и повторять запуск. Так мы настраиваем промпт на этой выборке;
полученный результат ещё не показывает качество на новых страницах.

In [ ]:
# Входы модели не содержат правильных ответов.
model_inputs = cards[["sample_id","image_path"]].copy()
model_inputs["image_path"] = model_inputs["image_path"].map(lambda p: str(ROOT / p))
gold = cards[["sample_id","reference_price"]].copy()
display(model_inputs.head(3))

Загружаем небольшую VLM. В Colab включите GPU; CPU тоже работает, но медленнее.
`prepare_vlm_input` — та же функция подготовки изображения, которую мы написали выше.

In [ ]:
from workbook_model import HintModel
import torch
import time
import re
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HintModel(device=device)
print("Устройство:",device)

In [ ]:
def predict_price(image_path, prompt):
    picture, messages = prepare_vlm_input(image_path,prompt)
    text = model.processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs = model.processor(text=[text],images=[picture],return_tensors="pt").to(model.device)
    with torch.inference_mode():
        generated = model.model.generate(**inputs,do_sample=False,max_new_tokens=96)
    answer_tokens = generated[:,inputs.input_ids.shape[1]:]
    raw = model.processor.batch_decode(answer_tokens,skip_special_tokens=True)[0]
    return raw

def generate_prices(prompt, inputs):
    rows = []
    for i,item in enumerate(inputs.itertuples(),1):
        raw = predict_price(item.image_path,prompt)
        row = {"sample_id":item.sample_id,"raw":raw,"model_price":np.nan,"error":None}
        try:
            text = re.sub(r"^```(?:json)?\s*|\s*```$","",raw.strip())
            parsed = json.loads(text)
            if set(parsed) != {"price","status"} or parsed["status"] not in {"ok","no_price","uncertain"}:
                raise ValueError("Неверная структура ответа")
            price = parsed["price"]
            if parsed["status"] == "ok":
                if type(price) not in (int,float) or not np.isfinite(price) or price <= 0:
                    raise ValueError("Нужна положительная цена")
                row["model_price"] = float(price)
            elif price is not None:
                raise ValueError("При отказе цена должна быть null")
        except (ValueError,TypeError,KeyError) as exc:
            row["error"] = str(exc)
        rows.append(row)
        print(f"{i}/{len(inputs)}",end=" · " if i % 10 else "\n")
    return pd.DataFrame(rows)

def evaluate_prices(predictions, gold):
    assert predictions.sample_id.is_unique and set(predictions.sample_id) == set(gold.sample_id)
    evaluated = predictions.merge(gold,on="sample_id",validate="one_to_one")
    evaluated["correct"] = (evaluated.model_price - evaluated.reference_price).abs() < .005
    return evaluated

In [ ]:
v1_predictions = generate_prices(PROMPT_V1,model_inputs)
v1 = evaluate_prices(v1_predictions,gold)
v1_accuracy = v1["correct"].mean()
display(Markdown(f"**V1: {v1_accuracy:.1%} верных цен** ({int(v1.correct.sum())}/{len(v1)})."))

**Перед изменением промпта посмотрим ошибки.** Какие числа выбрала V1?
Есть ли они на странице и к каким условиям относятся? Выберите до трех карточек из таблицы для просмотра.

Сначала разделим **наблюдаемые исходы**, не угадывая причины: ошибка формата, ответ без цены,
заниженная цена или завышенная. По типу страницы отдельно посмотрим, где V1 ошибается чаще.
Тип страницы «рассрочка» еще не означает, что модель выбрала платеж рассрочки.

Для подробностей в следующей ячейке задайте `selected_error_type` и `selected_ids`.
Скопируйте ID из таблицы или оставьте `selected_ids = []`, чтобы увидеть первые три ошибки.
Вывод содержит URL, исходный JSON модели и комментарий к gold — картинки автоматически не открываются.

In [ ]:
# Категории, которые можно определить из ответа без догадок о механизме ошибки.
error_analysis = v1.merge(cards[["sample_id","source_url","trap_type","reference_reason"]],on="sample_id",validate="one_to_one")
error_analysis["error_type"] = np.select([
    error_analysis["correct"],
    error_analysis["error"].notna(),
    error_analysis["model_price"].isna(),
    error_analysis["model_price"] < error_analysis["reference_price"],
], ["Верно","Ошибка формата","Ответ без цены","Цена ниже gold"],default="Цена выше gold")
outcome_counts = error_analysis["error_type"].value_counts()
display(outcome_counts.to_frame("Карточек"))

page_names = {"wholesale":"Опт","installment":"Рассрочка","variant":"Вариант товара", "package":"Упаковка / единица",
              "registration":"Регистрация","buyer_type":"Тип покупателя","old_current":"Старая / текущая цена",
              "quantity_discount":"Скидка за количество","online_store":"Онлайн / офлайн","single":"Одна цена"}
page_quality = error_analysis.groupby("trap_type").agg(Карточек=("correct","size"),Точность=("correct","mean"))
display(page_quality.rename(index=page_names).style.format({"Точность":"{:.1%}"}))
failures = error_analysis.loc[~error_analysis["correct"]].copy()
display(failures[["sample_id","error_type","model_price","reference_price"]].rename(columns={
    "sample_id":"ID карточки","error_type":"Исход","model_price":"V1, ₽","reference_price":"Gold, ₽"}))

In [ ]:
# Выберите категорию из таблицы выше или оставьте None для всех ошибок.
selected_error_type = None  # Например: "Цена ниже gold"
# Вставьте реальные ID из таблицы. Пустой список = первые три в выбранной категории.
selected_ids = []
selected = failures if selected_error_type is None else failures.loc[failures.error_type.eq(selected_error_type)]
if selected_ids:
    unknown_ids = set(selected_ids) - set(selected.sample_id)
    if unknown_ids:
        raise ValueError(f"Эти ID не входят в выбранную категорию ошибок: {sorted(unknown_ids)}")
    selected = selected.set_index("sample_id").loc[selected_ids].reset_index()
else:
    selected = selected.head(3)

import html
for row in selected.itertuples():
    display(Markdown(f"**{row.sample_id}: {row.error_type}. V1: {row.model_price:g} ₽; gold: {row.reference_price:g} ₽.**"))
    display(HTML(f'<a href="{html.escape(row.source_url,quote=True)}" target="_blank" rel="noopener noreferrer">Открыть страницу ↗</a>'))
    print("Ответ модели:",row.raw)
    print("Комментарий к gold:",row.reference_reason)
if selected.empty:
    print("Ошибок в выбранной категории нет.")

### Практическое задание №5. Изменим промпт по реальным ошибкам V1

Мы уже увидели ошибки V1. Напишите `PROMPT_V2`: свяжите цену с нужным товаром и вариантом,
различайте обычные и специальные цены, учитывайте условия их получения.
Не принимайте опт, платёж рассрочки или другую комплектацию за нужную цену.

Можно использовать ту же инструкцию `TASK_RULE`, которую получает человек.
Не меняйте модель, изображения и формат JSON; не передавайте gold и не подставляйте правильные числа.
Сформулируйте, какие ошибки вы ожидаете исправить и какие новые ошибки могут появиться.

**Мы проектируем V2 после просмотра ошибок V1 на этих же карточках. Поэтому используем их как dev-набор,**
а не как независимый тест обобщения.
Разумный на вид промпт не гарантирует улучшения — это предстоит измерить.

In [ ]:
# 1. Свяжите цену с товаром и вариантом.
# 2. Добавьте правила для специальных цен; можно использовать TASK_RULE.
# 3. Сохраните формат JSON price/status. Не вписывайте правильные цены.
PROMPT_V2 = ...  # YOUR CODE HERE: строка с новой инструкцией
print(PROMPT_V2)

**Что делает следующий запуск?**

1. `generate_prices` отправляет каждый скриншот в VLM с `PROMPT_V2` и получает JSON с ценой и статусом.
2. `evaluate_prices` соединяет ответы с gold и помечает правильность каждой цены.
3. `mean()` считает accuracy по всем карточкам, включая ошибки формата и ответы без цены.

Это генерация модельных ответов и оценка **модели**, не повторная разметка людьми.
Для показа исполнителю текст подсказки формируется из JSON функцией `render_hint`.
Изменили промпт — повторите запуск и график сравнения.

In [ ]:
# 1. Новые ответы VLM на все скриншоты с PROMPT_V2.
v2_predictions = generate_prices(PROMPT_V2,model_inputs)
# 2. Сравнение каждой цены с golden set.
v2 = evaluate_prices(v2_predictions,gold)
# 3. Доля правильных ответов модели.
v2_accuracy = v2["correct"].mean()

Сравним качество на golden set и переходы между верными/неверными ответами.
Смотрим не только исправления, но и новые ошибки. Затем можно вернуться к V2 и проверить следующую гипотезу.

In [ ]:
# 1. Две доли верных цен из текущих генераций.
model_accuracy = pd.Series({"V1":v1_accuracy,"V2":v2_accuracy})
# 2. Соединяем ответы по ID, а не по порядку строк.
paired = v1[["sample_id","correct"]].merge(v2[["sample_id","correct"]],on="sample_id",suffixes=("_v1","_v2"),validate="one_to_one")
fixed = int((~paired.correct_v1 & paired.correct_v2).sum())
regressed = int((paired.correct_v1 & ~paired.correct_v2).sum())
change_pp = 100 * (v2_accuracy - v1_accuracy)
# 3. Графики используют ровно эти вычисления.
fig, axes = plt.subplots(1,2,figsize=(11,3.5),layout="constrained")
bars = axes[0].bar(model_accuracy.index,100*model_accuracy,color=COLORS)
axes[0].bar_label(bars,labels=[f"{p:.1%}" for p in model_accuracy],padding=4)
axes[0].set(ylim=(0,110),ylabel="Верных цен, %",title=f"V2 − V1: {change_pp:+.1f} п.п.")
bars = axes[1].bar(["Исправили","Испортилось"],[fixed,regressed],color=["#3A9C89","#D76D67"])
axes[1].bar_label(bars,padding=4)
axes[1].set(ylabel="Карточек",title="Что скрывается за общей разницей?",ylim=(0,max(fixed,regressed,1)*1.3))
plt.show()

## Что можем заключить?

Для человеческого эксперимента проверим интервалы тем же способом: повторно набираем исполнителей,
а карточки оставляем фиксированными. Интервалы не описывают результат на всех интернет-магазинах.
Качество подсказки задаёт подвыборку, а не отдельное случайное назначение: различия этих срезов описательные.
Gold относится к сохранённому состоянию страницы; сайт и цена со временем могут измениться.

In [ ]:
human_intervals = pd.DataFrame({
    "Совпадение с неверной подсказкой":cluster_interval(data,"matches_hint",subset="wrong"),
    "Общая правильность":cluster_interval(data,"correct"),
}).T * 100
fig, ax = plt.subplots(figsize=(9,3))
for y,(label,row) in enumerate(human_intervals.iterrows()):
    ax.hlines(y,row["low"],row["high"],lw=5,color="#6479AF")
    ax.scatter(row["effect"],y,color="#D98559",s=70,zorder=3)
ax.axvline(0,color="black",ls="--")
ax.set(yticks=range(len(human_intervals)),yticklabels=human_intervals.index,
       xlabel="AI hint − Baseline, п.п.",title="95% интервалы: фиксированная выборка карточек")
plt.show()

| Мы узнали | Мы не узнали |
|---|---|
| С подсказкой люди чаще выбирали предложенную ошибочную цену | Скопировал ли подсказку конкретный человек |
| Верные подсказки помогли, неверные навредили; на нашей выборке помощь перевесила | Каким будет итог на другой выборке страниц и ошибок |
| Наблюдаемая медиана времени выросла, но интервал включает ноль | Есть ли устойчивое замедление или ускорение |

Общий выигрыш и локальный вред не противоречат друг другу. Итог зависит от состава выборки.

**Мы изменили промпт и измерили качество VLM. Что теперь можем сказать о качестве разметки людей?
Какие выводы семинара поддерживают ваш ответ и какое измерение вы предложите следующим?**

# Дополнительный анализ

## A. Направление и размер ошибки

$$r_{ij}=\frac{H_{ij}-Y_i}{Y_i},\qquad a_{ij}=|r_{ij}|.$$

$r<0$ — цену занизили, $r>0$ — завысили. $a$ — величина ошибки без направления.

**Задание.** На карточках с неверной подсказкой:

1. `numeric_answers`: оставьте числовые ответы, не превращайте отказы в ноль.
2. `price_difference`: ответ `price` минус gold `reference_price`.
3. Столбец `relative_error`: разделите разность на gold; `absolute_relative_error`: возьмите модуль.

После заполнения график покажет накопленную долю ответов при каждом значении ошибки.
Около нуля стоят правильные цены, слева — заниженные.

In [ ]:
# 1. Здесь анализируем только числовые ответы на wrong-hint подвыборке.
numeric_answers = ...  # YOUR CODE HERE: только numeric == True
# 2. Отклонение в рублях.
price_difference = ...  # YOUR CODE HERE: H − Y
# 3. Отклонение относительно стоимости товара и его модуль.
numeric_answers["relative_error"] = ...  # YOUR CODE HERE: (H − Y) / Y
numeric_answers["absolute_relative_error"] = ...  # YOUR CODE HERE: модуль относительной ошибки

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
for arm,label,color in zip(ARMS,LABELS,COLORS):
    values = np.sort(numeric_answers.loc[numeric_answers.condition.eq(arm),"relative_error"])
    cumulative_share = np.arange(1,len(values)+1) / len(values)
    ax.step(values,cumulative_share,where="post",label=f"{label}, n={len(values)}",color=color)
ax.axvline(0,color="black",ls="--")
ax.set(xlabel="Относительная ошибка цены",ylabel="Накопленная доля ответов",xscale="symlog",title="Занижают или завышают цену?")
ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(); plt.show()
error_summary = numeric_answers.groupby("condition").agg(
    answers=("relative_error","size"),signed_mean=("relative_error","mean"),absolute_mean=("absolute_relative_error","mean"))
display(error_summary.reindex(ARMS).rename(index=dict(zip(ARMS,LABELS)),columns={"answers":"Числовых ответов","signed_mean":"Средняя ошибка со знаком","absolute_mean":"Средняя абсолютная ошибка"})
    .style.format({"Средняя ошибка со знаком":"{:.1%}","Средняя абсолютная ошибка":"{:.1%}"}))

## B. Что считать достаточно точной ценой?

$$Q_g(\tau)=\frac{1}{N_g}\sum_{i,j\in g}\mathbb{1}\left[\frac{|H_{ij}-Y_i|}{Y_i}\leq\tau\right].$$

$\tau$ — допустимая относительная ошибка. Отказы остаются в знаменателе и не проходят ни один допуск.
При нулевом допуске используем прежнее совпадение до половины копейки.

**Задание:** заполните функцию `accuracy_at_tolerance(tolerance)`.

1. `absolute_error`: модуль разности цены и gold, разделённый на gold, для всего `data`.
2. `within_tolerance`: сравните с `tolerance`, добавьте точные совпадения `correct` и оставьте только `numeric`.
3. `accuracy`: среднее этого булевого признака по `condition`.

Готовый код вызовет вашу функцию для нескольких допусков и построит интерактивный график.
Двигая ползунок, мы меняем определение допустимой ошибки, а не ответы людей.

In [ ]:
def accuracy_at_tolerance(tolerance):
    # 1. Относительная величина ошибки каждого ответа.
    absolute_error = ...  # YOUR CODE HERE: |H − Y| / Y
    # 2. Отказы всегда False; точные совпадения проходят даже нулевой допуск.
    within_tolerance = ...  # YOUR CODE HERE: numeric И (correct ИЛИ ошибка ≤ допуск)
    # 3. Знаменатель — все ответы каждой группы.
    accuracy = ...  # YOUR CODE HERE: средняя within_tolerance по condition
    return accuracy

In [ ]:
import plotly.graph_objects as go
from plotly.offline import plot
tolerances = [0,.001,.005,.01,.02,.05,.10,.20]
tolerance_accuracy = pd.DataFrame([accuracy_at_tolerance(t) for t in tolerances],index=tolerances)
fig = go.Figure(go.Bar(x=LABELS,y=tolerance_accuracy.iloc[0].tolist(),marker_color=COLORS,
    text=[f"{v:.1%}" for v in tolerance_accuracy.iloc[0]],textposition="outside"))
steps = []
for tolerance,row in tolerance_accuracy.iterrows():
    steps.append(dict(method="update",label=f"{tolerance:.1%}",args=[
        {"y":[row.tolist()],"text":[[f"{v:.1%}" for v in row]]},
        {"title":f"Допуск {tolerance:.1%}: какие ответы считаем достаточно точными?"}]))
fig.update_layout(title="Нулевой допуск: точная цена",yaxis=dict(range=[0,1.1],tickformat=".0%",title="Доля всех ответов"),
    sliders=[dict(active=0,currentvalue={"prefix":"Допуск: "},steps=steps)],height=420)
display(HTML(plot(fig,output_type="div",include_plotlyjs=True)))

## C. Согласие для числовой разметки: α Криппендорфа

$$\alpha=1-\frac{D_o}{D_e}.$$

Сравнивать квадраты разностей рублей напрямую неудобно: дорогие товары будут доминировать.
Возьмём $z=\log(H/Y)$ и расстояние $d(a,b)=(a-b)^2$.
Gold здесь задаёт масштаб товара, но **не добавляется как голос исполнителя**.
Это согласие относительных отклонений, а не согласие абсолютных рублёвых цен.

Для карточки $i$ с $n_i\geq2$ числовыми ответами и $N=\sum_i n_i$:

$$D_o=\frac{1}{N}\sum_i\frac{\sum_{j\ne k}(z_{ij}-z_{ik})^2}{n_i-1},
\qquad D_e=2s_z^2.$$

Внутри суммы — упорядоченные пары, $s_z^2$ — выборочная дисперсия всех участвующих $z$ (`ddof=1`).

**Задание:** заполните шаги в `price_alpha(group)`.

1. Числовые положительные ответы переведите в `log_ratio = log(price/reference_price)`.
2. Готовый код соберёт `units`: по массиву на карточку, минимум два ответа.
3. `pair_distances`: матрица квадратов попарных разностей на карточке.
4. `observed_disagreement`: взвешенная сумма из формулы / `n_values`.
5. `expected_disagreement`: удвоенная дисперсия `pooled`.
6. `alpha`: подставьте две величины в формулу. При нулевом $D_e$ коэффициент не определён.

После графика обсудим: может ли согласие вырасти, если все стали одинаково ошибаться?
Отказы здесь исключены, поэтому рядом показываем, сколько карточек вошло в расчёт.

In [ ]:
def price_alpha(group):
    # 1. Нормируем числовые цены на стоимость товара.
    numeric = group.loc[group["numeric"]].copy()
    numeric["log_ratio"] = ...  # YOUR CODE HERE: log(H/Y)
    # 2. Одна карточка = один массив ответов. Нужно минимум два.
    units = [rows.log_ratio.to_numpy() for _,rows in numeric.groupby("sample_id") if len(rows) >= 2]
    if not units:
        return pd.Series({"alpha":np.nan,"cards":0,"D_o":np.nan,"D_e":np.nan})
    pooled = np.concatenate(units)
    n_values = len(pooled)
    weighted_disagreement = 0.0
    for values in units:
        # 3. Квадраты разностей всех упорядоченных пар (диагональ равна 0).
        pair_distances = ...  # YOUR CODE HERE: квадрат попарной разности
        weighted_disagreement += pair_distances.sum() / (len(values)-1)
    # 4. Наблюдаемое расхождение.
    observed_disagreement = ...  # YOUR CODE HERE: сумма / N
    # 5. Расхождение при случайном объединении ответов.
    expected_disagreement = ...  # YOUR CODE HERE: 2 × выборочная дисперсия
    # 6. При нулевой дисперсии alpha не определена.
    alpha = ...  # YOUR CODE HERE: 1 − D_o/D_e; NaN при D_e == 0
    return pd.Series({"alpha":alpha,"cards":len(units),"D_o":observed_disagreement,"D_e":expected_disagreement})

In [ ]:
# Для сравнения используем одинаковые карточки с ≥2 числовыми ответами в обеих группах.
numeric_counts = data.loc[data.numeric].groupby(["sample_id","condition"]).size().unstack(fill_value=0).reindex(columns=ARMS,fill_value=0)
eligible_ids = numeric_counts.index[numeric_counts.ge(2).all(axis=1)]
alpha_rows = {}
for arm in ARMS:
    alpha_group = data.loc[data.condition.eq(arm) & data.sample_id.isin(eligible_ids)]
    alpha_rows[arm] = price_alpha(alpha_group)
alpha_table = pd.DataFrame(alpha_rows).T.reindex(ARMS)
display(alpha_table.rename(index=dict(zip(ARMS,LABELS)),columns={"alpha":"α","cards":"Карточек"}).style.format({"α":"{:.3f}","Карточек":"{:.0f}","D_o":"{:.3f}","D_e":"{:.3f}"}))
fig, ax = plt.subplots(figsize=(7,3))
bars = ax.bar(LABELS,alpha_table["alpha"],color=COLORS)
ax.bar_label(bars,fmt="%.3f",padding=4)
ax.axhline(0,color="black",lw=1)
ax.set(ylabel="α для log(H/Y)",title="Согласие относительных ошибок — не accuracy",ylim=(min(0,alpha_table.alpha.min()-.1),1.1))
plt.show()

### Материалы

- Лекция 5: качество и время, α Криппендорфа, uptake и влияние подсказок, независимый контроль.
- [Roitero et al., 2025](https://arxiv.org/abs/2501.18265).
- [Варианты расстояния для α](https://arxiv.org/abs/2103.12170).

Состав подготовленного набора подсказок и история подготовки сохранены в исходных данных для воспроизводимости.
Не интерпретируем его как естественную частоту ошибок показанной VLM.